# 🧵 Fabric Defect Detection with YOLO (Classification)
Simple, clean pipeline: Extract dataset -> Train/Val Split -> Train YOLO -> Predict

In [1]:
import os, shutil, random, zipfile
from pathlib import Path
import torch
from ultralytics import YOLO

# 1. Device check
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [ ]:
# 2. Extract dataset from data/archive.zip
zip_path = Path('data/archive.zip')
extract_dir = Path('data/raw_dataset')

if not extract_dir.exists():
    print('Extracting dataset...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_dir)
    print('✓ Extracted!')
else:
    print('✓ Already extracted!')

In [ ]:
# 3. Simple Train (80%) / Val (20%) Split
src = Path('data/raw_dataset/Fabric Defects Dataset/Fabric Defect Dataset')
yolo_data = Path('data/yolo_dataset')

if yolo_data.exists():
    shutil.rmtree(yolo_data)

for folder in src.iterdir():
    if not folder.is_dir():
        continue
    
    cls_name = folder.name.replace(' ', '_').lower()
    imgs = list(folder.glob('*.jpg')) + list(folder.glob('*.png'))
    random.shuffle(imgs)
    
    split = int(0.8 * len(imgs))
    
    train_dir = yolo_data / 'train' / cls_name
    val_dir = yolo_data / 'val' / cls_name
    train_dir.mkdir(parents=True, exist_ok=True)
    val_dir.mkdir(parents=True, exist_ok=True)
    
    for img in imgs[:split]:
        shutil.copy(img, train_dir / img.name)
    for img in imgs[split:]:
        shutil.copy(img, val_dir / img.name)

print('✓ Train / Val split completed successfully!')

In [ ]:
# 4. Load Pretrained YOLO Model and Train
model = YOLO('yolo11n-cls.pt')

results = model.train(
    data=str(yolo_data.resolve()),
    epochs=25,
    imgsz=224,
    batch=16,
    device=device,
    project='fabric_qc',
    name='yolo_classifier'
)

In [ ]:
# 5. Evaluate Validation Accuracy
metrics = model.val()
print(f'Validation Top-1 Accuracy: {metrics.top1:.4f}')

In [ ]:
# 6. Test Prediction on a Random Image
val_images = list((yolo_data / 'val').glob('*/*.*'))
test_img = random.choice(val_images)

pred = model(str(test_img))
top1_idx = pred[0].probs.top1
predicted_label = pred[0].names[top1_idx]
confidence = pred[0].probs.top1conf.item()

print(f'Actual Class:    {test_img.parent.name}')
print(f'Predicted Class: {predicted_label} ({confidence*100:.2f}% confidence)')